# 02 — Dataset Relevance Analysis

**Project:** Ontology-Guided Hypothesis Generation Using LLMs and Topic Modeling in mHealth Research

This notebook analyzes the relevance filtering results and provides detailed insights
into how well the collected papers align with mHealth research.

**Date:** 2026-08-29

In [1]:
import pandas as pd
import numpy as np
import re
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
ANNOTATED_FILE = os.path.join(PROJECT_ROOT, 'data', 'processed', 'mhealth_relevance_annotated.csv')
RELEVANT_FILE = os.path.join(PROJECT_ROOT, 'data', 'processed', 'mhealth_relevant_dataset.csv')
LOW_REL_FILE = os.path.join(PROJECT_ROOT, 'data', 'processed', 'mhealth_low_relevance.csv')
FIGURES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'figures')
SUMMARIES_DIR = os.path.join(PROJECT_ROOT, 'reports', 'summaries')
os.makedirs(FIGURES_DIR, exist_ok=True)
os.makedirs(SUMMARIES_DIR, exist_ok=True)

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.2)
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['figure.dpi'] = 150

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

print('Setup complete.')

Setup complete.


In [2]:
# Load annotated dataset
df = pd.read_csv(ANNOTATED_FILE)
print(f'Total annotated papers: {len(df)}')
print(f'Columns: {df.columns.tolist()}')

# Clean year column — extract 4-digit year, handle NaN
def extract_year(val):
    if pd.isna(val):
        return 'Unknown'
    m = re.search(r'\d{4}', str(val))
    return m.group(0) if m else 'Unknown'

df['year_clean'] = df['year'].apply(extract_year)

Total annotated papers: 2628
Columns: ['pmid', 'title', 'abstract', 'year', 'mesh_terms', 'journal', 'authors', 'doi', 'document', 'cleaned_text', 'source', 'keyword_score', 'matched_keywords', 'mesh_score', 'matched_mesh', 'tfidf_score', 'relevance_score', 'relevance_category', 'relevance_reason']


## 1. Relevance Distribution

In [3]:
# Relevance category counts
cat_order = ['Highly relevant', 'Relevant', 'Possibly relevant', 'Low relevance']
cat_counts = df['relevance_category'].value_counts().reindex(cat_order).fillna(0).astype(int)

print('=== RELEVANCE DISTRIBUTION ===')
for cat, count in cat_counts.items():
    pct = 100 * count / len(df)
    print(f'  {cat}: {count} ({pct:.1f}%)')

# Pie chart and bar chart
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

colors = ['#2ecc71', '#3498db', '#f39c12', '#e74c3c']
explode = (0.05, 0.02, 0.02, 0.05)

axes[0].pie(cat_counts.values, labels=cat_counts.index, colors=colors,
            autopct='%1.1f%%', startangle=90, explode=explode,
            textprops={'fontsize': 10})
axes[0].set_title('Relevance Category Distribution', fontsize=14, fontweight='bold')

bars = axes[1].bar(cat_counts.index, cat_counts.values, color=colors, edgecolor='white')
axes[1].set_ylabel('Number of Papers')
axes[1].set_title('Papers by Relevance Category', fontsize=14, fontweight='bold')
for bar, val in zip(bars, cat_counts.values):
    axes[1].text(bar.get_x() + bar.get_width()/2., bar.get_height() + 10,
                 str(val), ha='center', va='bottom', fontweight='bold')
axes[1].tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'relevance_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

=== RELEVANCE DISTRIBUTION ===
  Highly relevant: 234 (8.9%)
  Relevant: 593 (22.6%)
  Possibly relevant: 923 (35.1%)
  Low relevance: 878 (33.4%)


Figure saved.


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_19552\1245259730.py:31: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 2. Relevance Score Distribution

In [4]:
print('=== RELEVANCE SCORE STATISTICS ===')
print(df['relevance_score'].describe().round(3))

fig, ax = plt.subplots(figsize=(12, 5))
ax.hist(df['relevance_score'], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
ax.axvline(0.7, color='green', linestyle='--', linewidth=2, label='Highly relevant (>=0.7)')
ax.axvline(0.5, color='blue', linestyle='--', linewidth=2, label='Relevant (>=0.5)')
ax.axvline(0.3, color='orange', linestyle='--', linewidth=2, label='Possibly relevant (>=0.3)')
ax.set_xlabel('Relevance Score')
ax.set_ylabel('Number of Papers')
ax.set_title('Distribution of Relevance Scores')
ax.legend(loc='upper right')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'relevance_score_distribution.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

=== RELEVANCE SCORE STATISTICS ===
count    2628.000
mean        0.411
std         0.193
min         0.036
25%         0.254
50%         0.397
75%         0.543
max         1.000
Name: relevance_score, dtype: float64


Figure saved.


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_19552\2101383303.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 3. Component Score Analysis

In [5]:
score_cols = ['keyword_score', 'mesh_score', 'tfidf_score', 'relevance_score']
print('=== COMPONENT SCORE STATISTICS ===')
print(df[score_cols].describe().round(3))

fig, ax = plt.subplots(figsize=(10, 6))
component_data = df[['keyword_score', 'mesh_score', 'tfidf_score']].melt(
    var_name='Component', value_name='Score'
)
component_data['Component'] = component_data['Component'].map({
    'keyword_score': 'Keyword\n(40% weight)',
    'mesh_score': 'MeSH\n(30% weight)',
    'tfidf_score': 'TF-IDF\n(30% weight)'
})
sns.boxplot(data=component_data, x='Component', y='Score', palette='Set2', ax=ax)
ax.set_title('Distribution of Scoring Components', fontsize=14, fontweight='bold')
ax.set_ylabel('Score (0-1)')
plt.tight_layout()
plt.savefig(os.path.join(FIGURES_DIR, 'component_scores_boxplot.png'), dpi=150, bbox_inches='tight')
plt.show()
print('Figure saved.')

=== COMPONENT SCORE STATISTICS ===
       keyword_score  mesh_score  tfidf_score  relevance_score
count       2628.000    2628.000     2628.000         2628.000
mean           0.361       0.610        0.277            0.411
std            0.324       0.265        0.155            0.193
min            0.000       0.100        0.010            0.036
25%            0.080       0.500        0.161            0.254
50%            0.320       0.500        0.251            0.397
75%            0.540       0.700        0.361            0.543
max            1.000       1.000        1.000            1.000
Figure saved.


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_19552\352929922.py:14: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `x` variable to `hue` and set `legend=False` for the same effect.

  sns.boxplot(data=component_data, x='Component', y='Score', palette='Set2', ax=ax)
C:\Users\VICTUS\AppData\Local\Temp\ipykernel_19552\352929922.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 4. Relevance by Year

In [6]:
# Relevance breakdown by year (using cleaned year)
year_rel = pd.crosstab(df['year_clean'], df['relevance_category'])
year_rel = year_rel.reindex(columns=cat_order, fill_value=0)

# Filter to valid 4-digit years
valid_years = sorted([y for y in year_rel.index if str(y).isdigit() and len(str(y)) == 4])

if valid_years:
    year_rel_valid = year_rel.loc[valid_years]
    print('=== RELEVANCE BY YEAR ===')
    print(year_rel_valid)
    
    fig, ax = plt.subplots(figsize=(14, 6))
    year_rel_valid.plot(kind='bar', stacked=True, color=colors, ax=ax, edgecolor='white')
    ax.set_xlabel('Publication Year')
    ax.set_ylabel('Number of Papers')
    ax.set_title('Relevance Categories by Publication Year', fontsize=14, fontweight='bold')
    ax.legend(title='Relevance', bbox_to_anchor=(1.02, 1), loc='upper left')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(os.path.join(FIGURES_DIR, 'relevance_by_year.png'), dpi=150, bbox_inches='tight')
    plt.show()
    print('Figure saved.')
else:
    print('No valid year data available for year-based relevance analysis.')
    print(f'Year values found: {df["year_clean"].unique()}')

=== RELEVANCE BY YEAR ===
relevance_category  Highly relevant  Relevant  Possibly relevant  \
year_clean                                                         
2012                              0         3                  3   
2018                             17        23                 41   
2019                             15        41                 41   
2020                             32        47                 74   
2021                             22        57                 86   
2022                             23        86                107   
2023                             29        76                108   
2024                             35        90                142   
2025                             32        69                 97   
2026                             12        27                 37   

relevance_category  Low relevance  
year_clean                         
2012                            0  
2018                           26  
2019         

Figure saved.


C:\Users\VICTUS\AppData\Local\Temp\ipykernel_19552\3878464591.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 5. Most Common Matched Keywords

In [7]:
all_kws = []
for kws in df['matched_keywords'].dropna():
    if str(kws).strip():
        for kw in str(kws).split(';'):
            kw = kw.strip()
            if kw:
                all_kws.append(kw)

kw_series = pd.Series(all_kws)
print(f'Total keyword matches: {len(kw_series)}')
print(f'\nTop 15 Matched Keywords:')
top_kws = kw_series.value_counts().head(15)
print(top_kws.to_string())

Total keyword matches: 5560

Top 15 Matched Keywords:
mobile health                1167
health app                    691
mhealth                       581
mobile health app             407
digital health                322
health application            316
mobile health application     274
wearable health               247
remote patient monitoring     219
telemedicine                  208
mobile healthcare             193
ehealth                       183
telehealth                    164
m-health                      159
wearable sensor                65


## 6. Examples from Each Category

In [8]:
for cat in cat_order:
    subset = df[df['relevance_category'] == cat]
    print(f'\n{"="*60}')
    print(f'{cat.upper()} - {len(subset)} papers')
    print(f'{"="*60}')
    if len(subset) > 0:
        sample = subset.sample(min(3, len(subset)), random_state=RANDOM_SEED)
        for _, row in sample.iterrows():
            print(f'\n  PMID: {row["pmid"]}')
            print(f'  Title: {row["title"][:120]}')
            print(f'  Score: {row["relevance_score"]:.3f}')
            print(f'  Reason: {row["relevance_reason"]}')


HIGHLY RELEVANT - 234 papers

  PMID: 40673359
  Title: Effectiveness of recommendations in promoting the use of mobile health applications in health guidance: a randomized con
  Score: 0.878
  Reason: Strong keyword match (1.00); Good MeSH alignment (1.00); Good semantic similarity (0.59)

  PMID: 38426885
  Title: Reducing the Environmental Impact of Digital Health: Development of an Ecoscore for Health Apps.
  Score: 0.781
  Reason: Strong keyword match (0.92); Good MeSH alignment (1.00); Good semantic similarity (0.38)

  PMID: 39377400
  Title: HealthCheck: A method for evaluating persuasive mobile health applications.
  Score: 0.797
  Reason: Strong keyword match (1.00); Good MeSH alignment (1.00); Good semantic similarity (0.32)

RELEVANT - 593 papers

  PMID: 37610798
  Title: A Nurse-Led Care Delivery App and Telehealth System for Patients Requiring Wound Care: Mixed Methods Implementation and 
  Score: 0.522
  Reason: Strong keyword match (0.76); No MeSH terms available; Goo

## 7. Low Relevance Analysis

In [9]:
df_low = df[df['relevance_category'] == 'Low relevance'].copy()
print(f'Low relevance papers: {len(df_low)}')
print(f'\nLow relevance score statistics:')
print(df_low['relevance_score'].describe().round(3))

borderline = df_low[(df_low['relevance_score'] >= 0.25) & (df_low['relevance_score'] < 0.30)]
print(f'\nBorderline cases (score 0.25-0.30): {len(borderline)}')
if len(borderline) > 0:
    print('\nBorderline examples (may deserve manual review):')
    for _, row in borderline.head(5).iterrows():
        print(f'  PMID: {row["pmid"]}, Score: {row["relevance_score"]:.3f}')
        print(f'  Title: {row["title"][:100]}')
        print(f'  Reason: {row["relevance_reason"]}')
        print()

Low relevance papers: 878

Low relevance score statistics:
count    878.000
mean       0.203
std        0.066
min        0.036
25%        0.166
50%        0.217
75%        0.254
max        0.300
Name: relevance_score, dtype: float64

Borderline cases (score 0.25-0.30): 251

Borderline examples (may deserve manual review):
  PMID: 42275060, Score: 0.280
  Title: Remote Monitoring Approaches to Reduce Readmissions After Infection and Sepsis: A Randomized Clinica
  Reason: Weak keyword match (0.08); Good MeSH alignment (0.70); Low semantic similarity (0.13)

  PMID: 42647085, Score: 0.261
  Title: Recruitment Source and Participant Retention in a Digital HIV Prevention Intervention for Young Blac
  Reason: Weak keyword match (0.16); MeSH score: 0.50; Good semantic similarity (0.16)

  PMID: 40773748, Score: 0.292
  Title: Cross-Platform Availability of Smartphone Sensors for Depression Indication Systems: Mixed-Methods U
  Reason: Weak keyword match (0.08); No MeSH terms available; Good s

## 8. Relevance Filtering Summary

In [10]:
df_relevant = pd.read_csv(RELEVANT_FILE)
print('=== RELEVANCE FILTERING SUMMARY ===')
print(f'Total papers analyzed: {len(df)}')
print(f'Papers retained (relevant): {len(df_relevant)}')
print(f'Papers removed (low relevance): {len(df_low)}')
print(f'\nRetained papers breakdown:')
retained_cats = df_relevant['relevance_category'].value_counts()
for cat, count in retained_cats.items():
    print(f'  {cat}: {count}')

print(f'\n--- METHODOLOGY NOTE ---')
print('Relevance was assessed using three complementary signals:')
print('1. Keyword matching (40%): mHealth domain terms in title and abstract')
print('2. MeSH term matching (30%): PubMed-assigned subject headings')
print('3. TF-IDF similarity (30%): semantic similarity to mHealth reference text')
print(f'\n--- LIMITATIONS ---')
print('- Papers without MeSH terms receive a neutral score (not penalized)')
print('- Novel mHealth applications may use different terminology')
print('- Borderline papers (score ~0.3) warrant manual review')
print('- The keyword list is comprehensive but not exhaustive')
print(f'\nRelevance analysis complete.')

=== RELEVANCE FILTERING SUMMARY ===


Total papers analyzed: 2628
Papers retained (relevant): 1750
Papers removed (low relevance): 878

Retained papers breakdown:
  Possibly relevant: 923
  Relevant: 593
  Highly relevant: 234

--- METHODOLOGY NOTE ---
Relevance was assessed using three complementary signals:
1. Keyword matching (40%): mHealth domain terms in title and abstract
2. MeSH term matching (30%): PubMed-assigned subject headings
3. TF-IDF similarity (30%): semantic similarity to mHealth reference text

--- LIMITATIONS ---
- Papers without MeSH terms receive a neutral score (not penalized)
- Novel mHealth applications may use different terminology
- Borderline papers (score ~0.3) warrant manual review
- The keyword list is comprehensive but not exhaustive

Relevance analysis complete.
